In [26]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2

In [3]:
plt.rcParams.update({
    "font.size": 16,          # default text size everywhere
    "axes.titlesize": 18,     # (optional) bigger title
    "axes.labelsize": 18,     # (optional) bigger axis labels
    "xtick.direction": "in",  # ticks point inward
    "ytick.direction": "in",
    "xtick.major.size": 6,    # major-tick length
    "ytick.major.size": 6,
    "xtick.minor.size": 3,    # minor-tick length
    "ytick.minor.size": 3,
    "legend.fontsize": 18,    # legend text
})

In [4]:
Jarzab_data1 = pd.read_excel('Jarzab Ecoli data/Ecoli R1-R3/results_TPP_TR_full_splitted_ids_links-1.xlsx')
Jarzab_data2 = pd.read_excel('Jarzab Ecoli data/Ecoli R1-R3/results_TPP_TR_full_splitted_ids_links-2.xlsx')
Jarzab_data3 = pd.read_excel('Jarzab Ecoli data/Ecoli R1-R3/results_TPP_TR_full_splitted_ids_links.xlsx')

In [5]:
Merged_Jarzab = pd.merge(Jarzab_data1,Jarzab_data2, on='gene_name',how='outer')
Merged_Jarzab = pd.merge(Merged_Jarzab,Jarzab_data3, on='gene_name',how='outer')

In [7]:


# Example: group replicate columns by temperature
replicate_groups = {
    "TMT126": ["norm_FC_TMT126_x", "norm_FC_TMT126_y", "norm_FC_TMT126"],
    "TMT127L": ["norm_FC_TMT127L_x", "norm_FC_TMT127L_y", "norm_FC_TMT127L"],
    "TMT127H": ["norm_FC_TMT127H_x", "norm_FC_TMT127H_y", "norm_FC_TMT127H"],
    "TMT128L": ["norm_FC_TMT128L_x", "norm_FC_TMT128L_y", "norm_FC_TMT128L"],
    "TMT128H": ["norm_FC_TMT128H_x", "norm_FC_TMT128H_y", "norm_FC_TMT128H"],
    "TMT129L": ["norm_FC_TMT129L_x", "norm_FC_TMT129L_y", "norm_FC_TMT129L"],
    "TMT129H": ["norm_FC_TMT129H_x", "norm_FC_TMT129H_y", "norm_FC_TMT129H"],
    "TMT130L": ["norm_FC_TMT130L_x", "norm_FC_TMT130L_y", "norm_FC_TMT130L"],
    "TMT130H": ["norm_FC_TMT130H_x", "norm_FC_TMT130H_y", "norm_FC_TMT130H"],
    "TMT131L": ["norm_FC_TMT131L_x", "norm_FC_TMT131L_y", "norm_FC_TMT131L"]
}

for temp, cols in replicate_groups.items():
    Merged_Jarzab[temp + "_median"] = Merged_Jarzab[cols].apply(lambda row: np.nanmedian(row.values.astype(float)), axis=1)


In [8]:
Merged_Jarzab_median = Merged_Jarzab[['Protein_ID','gene_name','TMT126_median','TMT127L_median','TMT127H_median',
                   'TMT128L_median','TMT128H_median','TMT129L_median','TMT129H_median',
                   'TMT130L_median','TMT130H_median','TMT131L_median']]

In [9]:

Temp = np.array([37,39.9,43,46.6,50.2,53.8,57.4,61,64.1,67])

In [24]:
def melting_curve(T, k, xo):
    T = np.array(T, dtype=float)
    return 1 / (1 + np.exp((k*T-k*xo)))

In [10]:
k_values = np.linspace(1, 20, 200)   
xo_values = np.linspace(1, 67, 500) 

In [11]:
def grid_search_with_sse_grid(pH, fold_changes, melting_curve,k_values, xo_values, downweight_thresh=None, downweight_factor=None):
    """
    Grid search over k and xo values to compute SSE grid and best fit.

    Parameters
    ----------
    pH : array_like
        Independent variable (x-axis, e.g., pH values).
    fold_changes : array_like
        Observed data (y-axis).
    melting_curve : callable(pH, k, xo) -> array_like
        Model function to compute predicted values.
    k_values : array_like
        Candidate k values.
    xo_values : array_like
        Candidate xo values.

    Returns
    -------
    dict with keys:
        - best_k : float, k with lowest SSE
        - best_xo : float, xo with lowest SSE
        - best_sse : float, minimum SSE found
        - sse_grid : 2D np.array of SSEs, shape (len(k_values), len(xo_values))
    """
    pH = np.asarray(pH)
    y = np.asarray(fold_changes)

    sse_grid = np.full((len(k_values), len(xo_values)), np.nan)
    best_sse = np.inf
    best_k, best_xo = None, None

     #Compute weights
    weights = np.ones_like(y)
    if downweight_thresh is not None:
        weights[y > downweight_thresh] = downweight_factor 


    for k_i, k in enumerate(k_values):
        for xo_j, xo in enumerate(xo_values):
            pred = melting_curve(pH, k, xo)
            
            residuals = y - pred
            # Weighted SSE
            sse = np.sum(weights * residuals**2)
            sse_grid[k_i, xo_j] = sse

            if sse < best_sse:
                best_sse = sse
                best_k, best_xo = k, xo

    return ( best_k, best_xo, best_sse, sse_grid)

In [12]:
def profile_ci_from_sse_grid(k_vals, xo_vals, sse_grid, best_k, best_xo, clean_pH, clean_y, model_func, n =10,alpha=0.05):
    """
    k_vals: 1D array of k grid
    xo_vals: 1D array of xo grid
    sse_grid: 2D array shape (len(k_vals), len(xo_vals)) with SSE values
    best_k, best_xo: best-fit parameters (should correspond near the min of sse_grid)
    clean_pH, clean_y: data to compute sigma2
    model_func: callable(x, k, xo)
    Returns dict with 1D CIs for k and xo from grid projection.
    """
    # compute sigma2 from best SSE (use the actual min in sse_grid)
    idx_min = np.unravel_index(np.nanargmin(sse_grid), sse_grid.shape)
    best_sse_grid = sse_grid[idx_min]
    p = 2
    sigma2 = best_sse_grid / max(1, n - p)
    dof = n-p
    sigma95 = dof * sigma2 / chi2.ppf(alpha/2, dof)  
    thresh = sigma95 * dof


    # For k: for each k (row), find minimum SSE over xo (i.e., profile)
    profile_xo = np.nanmin(sse_grid, axis=0)  # length len(xo_vals)

    # find where profile <= thresh and take interval
    def extract_interval(vals, profile):
        mask = profile <= thresh
        if not np.any(mask):
            # no crossing found; return NaNs
            return (np.nan, np.nan)
        # Use interpolation to estimate crossings at boundaries for more precision
        # Find contiguous region around the best value index
        best_idx = np.argmin(profile)
        # search left from best_idx to first index where profile > thresh
        left_idx = best_idx
        while left_idx > 0 and profile[left_idx] <= thresh:
            left_idx -= 1
        right_idx = best_idx
        while right_idx < len(profile)-1 and profile[right_idx] <= thresh:
            right_idx += 1

        # build local interpolation on [left_idx, right_idx]
        xi = vals[left_idx:right_idx+1]
        yi = profile[left_idx:right_idx+1]
        # ensure monotonic x for interp; if too small region, fallback to grid endpoints
        try:
            # interpolate crossing on left side
            if left_idx == 0 and yi[0] <= thresh:
                left_val = xi[0]
            else:
                f_left = interp1d(yi[:2], xi[:2], bounds_error=False, fill_value=(xi[0], xi[1]))
                # We want x where y == thresh -> invert: need interp of y->x
                # safer to use interpolation over (xi, yi) and invert with small dense grid
                dense_x = np.linspace(xi[0], xi[-1], 200)
                dense_y = np.interp(dense_x, xi, yi)
                left_cross = dense_x[np.where(dense_y <= thresh)[0][0]]
                left_val = left_cross
        except Exception:
            left_val = vals[left_idx]

        try:
            dense_x = np.linspace(xi[0], xi[-1], 200)
            dense_y = np.interp(dense_x, xi, yi)
            right_cross = dense_x[np.where(dense_y <= thresh)[0][-1]]
            right_val = right_cross
        except Exception:
            right_val = vals[right_idx]

        return (left_val, right_val)


    xo_ci = extract_interval(xo_vals, profile_xo)

    return xo_ci, thresh

In [20]:
def calculate_Tm_for_dataset(data, temp_values):

    protein_df = data[['gene_name', 'n','TMT126_median','TMT127L_median','TMT127H_median',
                       'TMT128L_median','TMT128H_median','TMT129L_median','TMT129H_median',
                       'TMT130L_median','TMT130H_median','TMT131L_median']]

    results = []

    for _, row in protein_df.iterrows():

        fold_changes = row.drop(labels=['gene_name', 'n']).values.astype(float)
        capped_fc = np.clip(fold_changes, None, 20)

        Temp = np.asarray(temp_values)
        mask = ~np.isnan(capped_fc)

        clean_T = Temp[mask]
        clean_fc = capped_fc[mask]

        if len(clean_T) < 4:
            results.append((row['gene_name'], np.nan, None))
            continue

        try:
            best_k, best_xo, best_sse, sse_grid = grid_search_with_sse_grid(
                clean_T,
                clean_fc,
                melting_curve,
                k_values,
                xo_values,
                downweight_factor=0,
                downweight_thresh=2.0
            )

            ci, thresh = profile_ci_from_sse_grid(
                k_values,
                xo_values,
                sse_grid,
                best_k,
                best_xo,
                clean_T,
                clean_fc,
                melting_curve,
                alpha=0.05,
                n=row['n']
            )

            Tm = best_xo
            params = (best_k, best_xo)

        except Exception as e:
            print(f"Fit failed for {row['gene_name']}: {e}")
            results.append((row['gene_name'], np.nan, None))
            continue

        results.append((row['gene_name'], Tm, ci, thresh,best_sse))

     

    return pd.DataFrame(results, columns=['gene_name', 'Tm', 'Tm_CI', 'thresh', 'best_sse'])

In [21]:
Jarzab_OO = pd.read_excel(r'Compare_Jarzab_OO model.xlsx')

In [22]:
Merged_Jarzab_median_n = Merged_Jarzab_median.merge(Jarzab_OO[['gene_name', 'n']], on='gene_name', how='right')

In [27]:
Tm_data =calculate_Tm_for_dataset(Merged_Jarzab_median_n, Temp)

/var/folders/xv/8x2vz_dj6j18tb3k4pkktjdh0000gn/T/ipykernel_71056/2273327370.py:3: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp((k*T-k*xo)))
/var/folders/xv/8x2vz_dj6j18tb3k4pkktjdh0000gn/T/ipykernel_71056/2273327370.py:3: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp((k*T-k*xo)))
/var/folders/xv/8x2vz_dj6j18tb3k4pkktjdh0000gn/T/ipykernel_71056/2273327370.py:3: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp((k*T-k*xo)))
/var/folders/xv/8x2vz_dj6j18tb3k4pkktjdh0000gn/T/ipykernel_71056/2273327370.py:3: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp((k*T-k*xo)))
/var/folders/xv/8x2vz_dj6j18tb3k4pkktjdh0000gn/T/ipykernel_71056/2273327370.py:3: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp((k*T-k*xo)))
/var/folders/xv/8x2vz_dj6j18tb3k4pkktjdh0000gn/T/ipykernel_71056/2273327370.py:3: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp((k*T-k*xo)))
/var/folders/xv/

In [29]:
Tm_data[['Tm_CI_lower', 'Tm_CI_upper']] = pd.DataFrame(
    Tm_data['Tm_CI'].tolist(),
    index=Tm_data.index)

In [ ]:
Tm_data.to_clipboard() # copied to Compare_Jazab_OO file